In [1]:
import pandas as pd
import numpy as np
import anndata as ad
import scanpy as sc
from GmGM import GmGM
from gprofiler import GProfiler
import warnings
import os
import shutil
from utilities import plot_graph
import matplotlib.pyplot as plt
from scipy import sparse, io, stats
import warnings
from pybiomart import Dataset
from sklearn import metrics
import matplotlib.transforms as transforms
from matplotlib.patches import Patch
from balanced_clustering import balanced_adjusted_mutual_info, balanced_homogeneity, balanced_completeness
import igraph as ig

In [6]:
adata = ad.read_h5ad('results/gmgm_pathway/anndata.h5ad')
bdata = adata.transpose()
G = adata.varp['var_gmgm_connectivities']

The transitivity of an Erdos-Renyi graph should be about the density of edges - we can confirm this below!

In [7]:
density = G.nnz / G.shape[0]**2
print(density)
result = 0
num = 100
for i in range(num):
    result += ig.Graph.Erdos_Renyi(n=G.shape[0], p=density).transitivity_undirected()
result / num

0.00031158383792246296


0.00030991495981939974

What about the modularity of clusterings?  Unlike for transitivity, there's no nice theoretical value, so we'll measure it empirically.

Coarse GmGM had a Leiden resolution of 0.29 so we could copy that, although we found it lead to modularities of 0!  And hence chose to use modularity of 1.

This will take a couple minutes.

In [20]:
results = []
for i in range(10):
    rG_ig = ig.Graph.Erdos_Renyi(n=G.shape[0], p=density)
    rG = rG_ig.get_adjacency_sparse()
    sc.tl.leiden(bdata, adjacency=rG, resolution=1, key_added='TEMP')
    results.append(rG_ig.modularity(bdata.obs['TEMP'].astype(int)))

/Users/baileyandrew/mambaforge/envs/cnr-colab/lib/python3.9/site-packages/igraph/community.py:506: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  return GraphBase.modularity(self, membership, weights, resolution, directed)
/Users/baileyandrew/mambaforge/envs/cnr-colab/lib/python3.9/site-packages/igraph/community.py:506: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  return GraphBase.modularity(self, membership, weights, resolution, directed)
/Users/baileyandrew/mambaforge/envs/cnr-colab/lib/python3.9/site-packages/igraph/community.py:506: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a 

In [21]:
np.mean(results), np.min(results), np.max(results)

(0.3359121866138236, 0.333886005267678, 0.337798669348405)

What about a sparser clustering?  We used a resolution of 3 for the original GmGM

In [22]:
results = []
for i in range(10):
    rG_ig = ig.Graph.Erdos_Renyi(n=G.shape[0], p=density)
    rG = rG_ig.get_adjacency_sparse()
    sc.tl.leiden(bdata, adjacency=rG, resolution=3, key_added='TEMP')
    results.append(rG_ig.modularity(bdata.obs['TEMP'].astype(int)))
np.mean(results), np.min(results), np.max(results)

/Users/baileyandrew/mambaforge/envs/cnr-colab/lib/python3.9/site-packages/igraph/community.py:506: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  return GraphBase.modularity(self, membership, weights, resolution, directed)
/Users/baileyandrew/mambaforge/envs/cnr-colab/lib/python3.9/site-packages/igraph/community.py:506: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  return GraphBase.modularity(self, membership, weights, resolution, directed)
/Users/baileyandrew/mambaforge/envs/cnr-colab/lib/python3.9/site-packages/igraph/community.py:506: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a 

(0.27141048448782346, 0.2701722625710287, 0.27226329898795987)

We can also see what the transitivity should be for a random hdWGCNA-based network

In [24]:
G = io.mmread("./results/gmgm_pathway/hdWGCNA_network.mtx").tocsc()

In [25]:
density = G.nnz / G.shape[0]**2
print(density)
result = 0
num = 100
for i in range(num):
    result += ig.Graph.Erdos_Renyi(n=G.shape[0], p=density).transitivity_undirected()
result / num

0.0003351431061063074


0.00033802325973605993

What about modularity of hdWGCNA-based networks?  Standard hdWGCNA was not clustered with Leiden (it was a built-in clustering algorithm to the hdWGCNA package), so we'll compare against default leiden (resolution=1).

In [35]:
# Reshape so the code below does not crash
sbdata = bdata[:G.shape[0]]

In [37]:
results = []
for i in range(10):
    rG_ig = ig.Graph.Erdos_Renyi(n=G.shape[0], p=density)
    rG = rG_ig.get_adjacency_sparse()
    sc.tl.leiden(sbdata, adjacency=rG, resolution=1, key_added='TEMP')
    results.append(rG_ig.modularity(sbdata.obs['TEMP'].astype(int)))
np.mean(results), np.min(results), np.max(results)

/Users/baileyandrew/mambaforge/envs/cnr-colab/lib/python3.9/site-packages/igraph/community.py:506: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  return GraphBase.modularity(self, membership, weights, resolution, directed)
/Users/baileyandrew/mambaforge/envs/cnr-colab/lib/python3.9/site-packages/igraph/community.py:506: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  return GraphBase.modularity(self, membership, weights, resolution, directed)
/Users/baileyandrew/mambaforge/envs/cnr-colab/lib/python3.9/site-packages/igraph/community.py:506: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a 

(0.4714977313459216, 0.4688405057737952, 0.47453489271764515)

Finally, what about fine hdWGCNA?

In [38]:
results = []
for i in range(10):
    rG_ig = ig.Graph.Erdos_Renyi(n=G.shape[0], p=density)
    rG = rG_ig.get_adjacency_sparse()
    sc.tl.leiden(sbdata, adjacency=rG, resolution=3.89155, key_added='TEMP')
    results.append(rG_ig.modularity(sbdata.obs['TEMP'].astype(int)))
np.mean(results), np.min(results), np.max(results)

/Users/baileyandrew/mambaforge/envs/cnr-colab/lib/python3.9/site-packages/igraph/community.py:506: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  return GraphBase.modularity(self, membership, weights, resolution, directed)
/Users/baileyandrew/mambaforge/envs/cnr-colab/lib/python3.9/site-packages/igraph/community.py:506: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  return GraphBase.modularity(self, membership, weights, resolution, directed)
/Users/baileyandrew/mambaforge/envs/cnr-colab/lib/python3.9/site-packages/igraph/community.py:506: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a 

(0.4467221311674269, 0.4424615277602031, 0.4501277353431023)